# House Prices - Model Inference

Load the best model from MLflow Model Registry, apply the same preprocessing pipeline, generate Kaggle submission.

## Setup

In [ ]:
import os
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import mlflow
import mlflow.sklearn
import dagshub

os.environ['MLFLOW_TRACKING_USERNAME'] = 'sansi23'
os.environ['MLFLOW_TRACKING_PASSWORD'] = '40232a6e77e82647fb1d4a6c4b838475b5ad16a6'

dagshub.init(
    repo_owner='sansi23',
    repo_name='House-Prices---Advanced-Regression-Techniques',
    mlflow=True
)

print('MLflow URI:', mlflow.get_tracking_uri())

## Load Best Model from Model Registry

In [ ]:
model_name    = 'house_prices_best_model'
model_version = 1

model_uri  = f'models:/{model_name}/{model_version}'
best_model = mlflow.sklearn.load_model(model_uri)

print('Model loaded:', type(best_model))
print('Model params:', best_model.get_params())

## Load & Preprocess Test Data

Apply the exact same steps as in `model_experiment.ipynb`:
1. Simple imputation
2. New features
3. OHE
4. Ordinal encoding
5. WOE encoding
6. Select RFE features

In [ ]:
train = pd.read_csv('data/train.csv')
test  = pd.read_csv('data/test.csv')

test_ids = test['Id']

X_train_raw = train.drop(columns=['SalePrice', 'Id'])
y_train     = np.log1p(train['SalePrice'])
X_test_raw  = test.drop(columns=['Id'])

print('X_train_raw:', X_train_raw.shape)
print('X_test_raw: ', X_test_raw.shape)

In [ ]:
# Simple imputation — same as experiment
def simple_impute(df_train, df_test):
    df_train = df_train.copy()
    df_test  = df_test.copy()
    for col in df_train.columns:
        if df_train[col].isnull().any():
            if df_train[col].dtype == 'object':
                fill_val = df_train[col].mode()[0]
            else:
                fill_val = df_train[col].median()
            df_train[col] = df_train[col].fillna(fill_val)
            df_test[col]  = df_test[col].fillna(fill_val)
    return df_train, df_test

X_train_s, X_test_s = simple_impute(X_train_raw, X_test_raw)
print('Nulls after imputation - train:', X_train_s.isnull().sum().sum())
print('Nulls after imputation - test: ', X_test_s.isnull().sum().sum())

In [ ]:
# New features
for df in [X_train_s, X_test_s]:
    df['TotalSF']      = df['TotalBsmtSF'] + df['1stFlrSF'] + df['2ndFlrSF']
    df['HouseAge']     = df['YrSold'] - df['YearBuilt']
    df['RemodAge']     = df['YrSold'] - df['YearRemodAdd']
    df['TotalBath']    = df['FullBath'] + df['HalfBath'] * 0.5 + df['BsmtFullBath'] + df['BsmtHalfBath'] * 0.5
    df['HasGarage']    = (df['GarageArea'] > 0).astype(int)
    df['HasFireplace'] = (df['Fireplaces'] > 0).astype(int)
    df['HasPool']      = (df['PoolArea'] > 0).astype(int)
    df['HasPorch']     = (df['OpenPorchSF'] + df['EnclosedPorch'] + df['3SsnPorch'] + df['ScreenPorch'] > 0).astype(int)

print('Shape after new features:', X_train_s.shape)

In [ ]:
# OHE
ohe_cols = [
    'MSZoning', 'Street', 'Alley', 'LotShape', 'LandContour', 'Utilities',
    'LotConfig', 'LandSlope', 'BldgType', 'HouseStyle', 'RoofStyle', 'RoofMatl',
    'Foundation', 'Heating', 'CentralAir', 'Electrical', 'Functional',
    'GarageType', 'GarageFinish', 'PavedDrive', 'Fence', 'MiscFeature',
    'SaleType', 'SaleCondition', 'BsmtExposure', 'BsmtFinType1', 'BsmtFinType2',
    'MasVnrType', 'Condition1', 'Condition2'
]

X_train_s = pd.get_dummies(X_train_s, columns=ohe_cols, drop_first=True)
X_test_s  = pd.get_dummies(X_test_s,  columns=ohe_cols, drop_first=True)
X_test_s  = X_test_s.reindex(columns=X_train_s.columns, fill_value=0)

print('Shape after OHE:', X_train_s.shape)

In [ ]:
# Ordinal encoding
qual_map  = {'Ex': 5, 'Gd': 4, 'TA': 3, 'Fa': 2, 'Po': 1, 'None': 0, 'NA': 0}
qual_cols = ['ExterQual', 'ExterCond', 'BsmtQual', 'BsmtCond',
             'HeatingQC', 'KitchenQual', 'FireplaceQu',
             'GarageQual', 'GarageCond', 'PoolQC']

for df in [X_train_s, X_test_s]:
    for col in qual_cols:
        df[col] = df[col].map(qual_map).fillna(0).astype(int)

print('Ordinal encoding done.')

In [ ]:
# WOE encoding
y_binary = (y_train > y_train.median()).astype(int)

def woe_encode(train_col, test_col, y_binary):
    woe_map = {}
    for cat in train_col.unique():
        mask       = train_col == cat
        events     = y_binary[mask].sum()
        non_events = (1 - y_binary[mask]).sum()
        total_ev   = y_binary.sum()
        total_nev  = (1 - y_binary).sum()
        dist_ev    = (events + 0.5) / total_ev
        dist_nev   = (non_events + 0.5) / total_nev
        woe_map[cat] = np.log(dist_ev / dist_nev)
    return train_col.map(woe_map).fillna(0), test_col.map(woe_map).fillna(0)

for col in ['Neighborhood', 'Exterior1st', 'Exterior2nd']:
    X_train_s[col], X_test_s[col] = woe_encode(X_train_s[col], X_test_s[col], y_binary)

print('WOE encoding done.')

In [ ]:
# Select the same RFE features the model was trained on
rfe_features = best_model.feature_names_in_.tolist()

X_test_final = X_test_s[rfe_features]

print('X_test_final shape:', X_test_final.shape)

## Generate Predictions & Submission

In [ ]:
log_predictions = best_model.predict(X_test_final)
predictions     = np.expm1(log_predictions)   # reverse log1p transform

submission = pd.DataFrame({
    'Id':        test_ids,
    'SalePrice': predictions
})

submission.to_csv('submission.csv', index=False)

print('Submission saved!')
print(submission.shape)
print(submission.head())